# BRD4–BD1 | Auditable biochemical IC₅₀ preprocessing

**Purpose:** derive one model-ready pIC₅₀ value per standardized molecule from ChEMBL-derived biochemical IC₅₀ records, without pooling BD1 with BD2. Follow the numbered cells from top to bottom. Every major filter prints a count and sample records.

**Source:** `BRD4_IC50_curated_preliminary.csv`, a previously extracted ChEMBL file containing 10,919 exact-IC₅₀ records. It is **not** the original 58,225-row unfiltered export. The assay-domain rules below are automated and need manual protein-construct/source verification before publication.

**Expected results for this input snapshot:** 685 assays → 228 accepted BD1 biochemical assays → 2,932 eligible assay measurements → 2,827 standardized structures → 11 flagged for cross-assay disagreement → **2,816 model-ready structures**.

**How to rerun:** install `pandas`, `numpy`, `rdkit`, and `IPython`; put the source CSV in `BD1/data/source/` when working from the repository root. All outputs are saved in `BD1/audit/` and `BD1/data/`.

## 1. Locate the input and inspect the raw extract

First verify the source exists; do **not** silently substitute another BRD4 cohort. The only absolute path below is a fallback for the original execution environment, not a requirement for other machines.

In [1]:
from pathlib import Path
import json, re
import numpy as np
import pandas as pd
from IPython.display import display
from rdkit import Chem, RDLogger
RDLogger.DisableLog('rdApp.*')
from rdkit.Chem.MolStandardize import rdMolStandardize

source_candidates = [
    Path('BD1/data/source/BRD4_IC50_curated_preliminary.csv'),
    Path('data/source/BRD4_IC50_curated_preliminary.csv'),
    Path('BRD4_IC50_curated_preliminary.csv'),
    Path('/mnt/data/files/BRD4_IC50_curated_preliminary.csv'), # execution-environment fallback
]
INPUT_CSV = next((p for p in source_candidates if p.is_file()), None)
if INPUT_CSV is None:
    raise FileNotFoundError('Place BRD4_IC50_curated_preliminary.csv in BD1/data/source/ before executing.')
# Choose output relative to the notebook location or repository root.
ROOT = Path('BD1') if Path('BD1').is_dir() else Path('.')
AUDIT_DIR = ROOT / 'audit'
DATA_DIR = ROOT / 'data'
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
raw = pd.read_csv(INPUT_CSV, low_memory=False)
required = ['Molecule ChEMBL ID','Smiles','Assay ChEMBL ID','Assay Description',
            'Assay Type','BAO Label','Document ChEMBL ID','Standard Type',
            'Standard Units','Standard Relation','Standard Value',
            'Data Validity Comment','Potential Duplicate']
missing = sorted(set(required) - set(raw.columns))
assert not missing, f'Missing source columns: {missing}'
print('Source:', INPUT_CSV.name)
print('Raw records:', len(raw), '| Columns:', len(raw.columns),
      '| Unique assay IDs:', raw['Assay ChEMBL ID'].nunique(),
      '| Unique molecule IDs:', raw['Molecule ChEMBL ID'].nunique())
display(raw[['Molecule ChEMBL ID','Assay ChEMBL ID','Standard Type',
             'Standard Value','Standard Units','Smiles']].head(5))

Source: BRD4_IC50_curated_preliminary.csv
Raw records: 10919 | Columns: 51 | Unique assay IDs: 685 | Unique molecule IDs: 7966


,Molecule ChEMBL ID,Assay ChEMBL ID,Standard Type,Standard Value,Standard Units,Smiles
0,CHEMBL5874820,CHEMBL5730426,IC50,193.0,nM,O=C(NO)c1ccc(-c2csc3c(=O)cc(N4CCOCC4)oc23)cc1
1,CHEMBL3949043,CHEMBL5730180,IC50,42.0,nM,Cc1cc(-c2nc3cnc(N4CCOC[C@@H]4C)cc3n2Cc2ccccc2)...
2,CHEMBL3949005,CHEMBL5730180,IC50,192.0,nM,CN1CCN(c2ccc3nc(-c4ccc(=O)n(C)n4)n(Cc4ccccc4)c...
3,CHEMBL3965053,CHEMBL5730180,IC50,55.0,nM,CCn1cc(-c2nc3cnc(N4CCN(C)CC4)cc3n2Cc2ccccc2)cc...
4,CHEMBL3972402,CHEMBL5730180,IC50,16.0,nM,Cc1cc(-c2nc3cc(C4CCOCC4)cnc3n2Cc2ccccc2)cn(C)c1=O


## 2. Inspect missingness, duplicates and endpoint formats

This is **diagnostic**: the table reports input quality, not exclusions. The next sections make the actual decisions.

In [2]:
qc = pd.DataFrame({'column':required, 'missing_n':[int(raw[x].isna().sum()) for x in required]})
qc['missing_pct'] = (100 * qc.missing_n / len(raw)).round(2)
display(qc.sort_values('missing_n',ascending=False))
for name in ['Standard Type','Standard Units','Assay Type','Standard Relation','Potential Duplicate']:
    print('\n'+name+':')
    display(raw[name].value_counts(dropna=False).rename_axis('value').reset_index(name='records').head(12))
print('Exact duplicate input rows:', int(raw.duplicated().sum()))

,column,missing_n,missing_pct
11,Data Validity Comment,10919,100.0
1,Smiles,0,0.0
2,Assay ChEMBL ID,0,0.0
3,Assay Description,0,0.0
0,Molecule ChEMBL ID,0,0.0
4,Assay Type,0,0.0
5,BAO Label,0,0.0
7,Standard Type,0,0.0
6,Document ChEMBL ID,0,0.0
8,Standard Units,0,0.0



Standard Type:


,value,records
0,IC50,10919



Standard Units:


,value,records
0,nM,10919



Assay Type:


... (2 outputs left out)

,value,records
0,'=',10919



Potential Duplicate:


,value,records
0,0,10919


Exact duplicate input rows: 3


## 3. Identify BD1-specific biochemical assays

**Accept** an assay only when its description explicitly names BD1 and the assay type is biochemical (`B`). **Review or exclude** descriptions involving BD2/mixed domains, mutants, cell formats, DSF/thermal-shift, or ambiguous domain annotations. Automated text classification is conservative, not a substitute for manual construct-level review.

**Why assay-level?** Measurements from the same assay must have consistent domain assignment; the record-level step later handles numerical validity.

In [3]:
BD1 = re.compile(r'(?<!\w)BD[\s\-_]?1(?!\w)|\bBDI\b|\bfirst bromodomain\b|\bbromodomain\s*(?:domain\s*)?1\b|\bBRD4\s*\(\s*1\s*\)',re.I)
BD2 = re.compile(r'(?<!\w)BD[\s\-_]?2(?!\w)|\bBDII\b|\bsecond bromodomain\b|\bbromodomain\s*(?:domain\s*)?2\b|\bBRD4\s*\(\s*2\s*\)',re.I)
DSF = re.compile(r'\bDSF\b|thermal shift|melting temperature|protein stability shift|differential scanning fluorimetry',re.I)
CELL = re.compile(r'\bcell(?:ular)? (?:viability|proliferation|growth|death|inhibition|IC50|assay|line)s?\b|cytotoxicity|anti[- ]?proliferat|tumou?r cell|\bMTT\b|\bCCK[- ]?8\b',re.I)
MUT = re.compile(r'\bmutant\b|\bmutation\b|\bY390A\b|\bY97A\b|\bvariant\b',re.I)
def classify_assay(row):
    description = str(row['Assay Description'] or '')
    if not BD1.search(description): return 'review_BD1_not_explicit'
    if row['Assay Type'] != 'B': return 'review_non_biochemical'
    if DSF.search(description): return 'exclude_DSF_thermal_shift'
    if CELL.search(description) or str(row['BAO Label']).lower() in (
        'cell-based format','organism-based format','tissue-based format'):
        return 'review_cellular_or_nonbiochemical_format'
    if MUT.search(description): return 'review_mutant_or_multi_construct'
    if BD2.search(description): return 'review_BD1_BD2_mixed_description'
    return 'accept_BD1_biochemical'

assays = raw.sort_values('Assay ChEMBL ID').drop_duplicates('Assay ChEMBL ID').copy()
assays['decision'] = assays.apply(classify_assay,axis=1)
assay_audit = assays[['Assay ChEMBL ID','Assay Description','Assay Type',
                      'BAO Label','Document ChEMBL ID','decision']].copy()
assay_audit['n_records_in_input'] = assay_audit['Assay ChEMBL ID'].map(
    raw['Assay ChEMBL ID'].value_counts())
display(assay_audit.groupby('decision',dropna=False).agg(
    assays=('Assay ChEMBL ID','size'),
    source_records=('n_records_in_input','sum')).reset_index().sort_values('assays',ascending=False))
print('Accepted BD1 assays:', int(assay_audit.decision.eq('accept_BD1_biochemical').sum()),
      '/',len(assay_audit))
assay_audit.to_csv(AUDIT_DIR/'BD1_assay_audit.csv', index=False)

,decision,assays,source_records
2,review_BD1_not_explicit,361,5445
0,accept_BD1_biochemical,228,2932
1,review_BD1_BD2_mixed_description,41,1456
4,review_mutant_or_multi_construct,30,582
3,review_cellular_or_nonbiochemical_format,23,457
5,review_non_biochemical,2,47


Accepted BD1 assays: 228 / 685


## 4. Apply IC₅₀ measurement-quality rules

Require `IC50`, units `nM`, equality relation, positive concentration, no validity warning, and no duplicate flag. Keep record-level exclusion reasons even for assays rejected earlier; the audit lets readers trace why each original row did not enter modeling.

In [4]:
raw['assay_decision'] = raw['Assay ChEMBL ID'].map(
    dict(zip(assays['Assay ChEMBL ID'], assays['decision'])))
raw['standard_value_nM'] = pd.to_numeric(raw['Standard Value'], errors='coerce')
quality = (raw['Standard Type'].eq('IC50') &
           raw['Standard Units'].eq('nM') &
           raw['Standard Relation'].astype(str).str.strip().isin(['=', "'='"]) &
           raw['standard_value_nM'].gt(0) &
           raw['Data Validity Comment'].isna() &
           raw['Potential Duplicate'].isin([0, False, np.nan]))
raw['eligibility'] = np.where(raw['assay_decision']=='accept_BD1_biochemical',
                              np.where(quality,'eligible','exclude_measurement_quality'),
                              raw['assay_decision'])
record_columns = ['Molecule ChEMBL ID','Smiles','Assay ChEMBL ID','Document ChEMBL ID',
                  'Assay Description','Assay Type','BAO Label','Standard Value',
                  'Standard Units','Standard Relation','Data Validity Comment',
                  'Potential Duplicate','eligibility']
raw[record_columns].to_csv(AUDIT_DIR/'BD1_record_decisions_all.csv',index=False)
record_counts = raw.eligibility.value_counts().rename_axis('decision').reset_index(name='records')
display(record_counts)
eligible = raw.loc[raw.eligibility=='eligible'].copy()
print(f'Records: {len(raw):,} input → {len(eligible):,} biochemical BD1 measurements')

,decision,records
0,review_BD1_not_explicit,5445
1,eligible,2932
2,review_BD1_BD2_mixed_description,1456
3,review_mutant_or_multi_construct,582
4,review_cellular_or_nonbiochemical_format,457
5,review_non_biochemical,47


Records: 10,919 input → 2,932 biochemical BD1 measurements


## 5. Standardize molecules and calculate pIC₅₀

RDKit parses SMILES, performs cleanup and removes disconnected counterions using `FragmentParent`. Stereochemical information is retained. Potency is calculated as **pIC₅₀ = 9 − log₁₀(IC₅₀ in nM)**; invalid structures are not assigned made-up values.

In [5]:
def standardize(smiles):
    try:
        mol=Chem.MolFromSmiles(str(smiles))
        if mol is None: return None
        mol=rdMolStandardize.Cleanup(mol)
        mol=rdMolStandardize.FragmentParent(mol)
        return Chem.MolToSmiles(mol,isomericSmiles=True)
    except Exception:
        return None
eligible['standardized_smiles'] = eligible['Smiles'].map(standardize)
eligible['pIC50'] = 9-np.log10(eligible['standard_value_nM'])
eligible[['Molecule ChEMBL ID','Smiles','standardized_smiles',
          'Assay ChEMBL ID','Document ChEMBL ID','standard_value_nM',
          'pIC50','Assay Description']].to_csv(AUDIT_DIR/'BD1_eligible_activity_records.csv',index=False)
good=eligible.loc[eligible.standardized_smiles.notna()].copy()
print('Eligible measurements:',len(eligible),
      '| valid structures:',len(good),
      '| invalid structure rows:',int(eligible.standardized_smiles.isna().sum()),
      '| unique standardized structures:',good.standardized_smiles.nunique())
display(good[['Smiles','standardized_smiles','standard_value_nM','pIC50']].head(5))

Eligible measurements: 2932 | valid structures: 2932 | invalid structure rows: 0 | unique standardized structures: 2827


,Smiles,standardized_smiles,standard_value_nM,pIC50
19,CCS(=O)(=O)c1ccc(Oc2cccc(NS(=O)(=O)c3ccccc3F)c...,CCS(=O)(=O)c1ccc(Oc2cccc(NS(=O)(=O)c3ccccc3F)c...,240.00,6.619789
20,Cn1cc(-c2cc(F)cc(NS(C)(=O)=O)c2)c2ccccc2c1=O,Cn1cc(-c2cc(F)cc(NS(C)(=O)=O)c2)c2ccccc2c1=O,2750.00,5.560667
23,CCn1c2c(c(=O)[nH]c1=O)C(c1ccc(C)cc1)C1=C(COC1=...,CCn1c2c(c(=O)[nH]c1=O)C(c1ccc(C)cc1)C1=C(COC1=...,1000.00,6.000000
24,CCC(=O)N1CCCc2cc(NS(=O)(=O)c3ccc(F)cc3C)ccc21,CCC(=O)N1CCCc2cc(NS(=O)(=O)c3ccc(F)cc3C)ccc21,74000.00,4.130768
25,CO[C@H]1CNC[C@@H](OCC2CCC(F)(F)CC2)[C@@H]1Nc1n...,CO[C@H]1CNC[C@@H](OCC2CCC(F)(F)CC2)[C@@H]1Nc1n...,25118.86,4.600000


## 6. Aggregate multiple IC₅₀ measurements without cross-assay weighting bias

For a molecule measured repeatedly in the same assay, calculate the **median pIC₅₀ within each assay** first. Then calculate the **median of those assay medians** to obtain a compound-level target. This avoids allowing assays with many repeat readings to dominate the compound value. Keep ChEMBL molecule, assay, and document identifiers.

In [6]:
def join_unique(values):
    return ';'.join(sorted(set(str(v) for v in values if pd.notna(v))))
per_assay=good.groupby(['standardized_smiles','Assay ChEMBL ID'],as_index=False).agg(
    assay_median_pIC50=('pIC50','median'),n_measurements=('pIC50','size'))
agg=per_assay.groupby('standardized_smiles',as_index=False).agg(
    pIC50=('assay_median_pIC50','median'),
    n_assays=('Assay ChEMBL ID','nunique'),
    min_assay_pIC50=('assay_median_pIC50','min'),
    max_assay_pIC50=('assay_median_pIC50','max'))
agg['interassay_pIC50_range']=agg.max_assay_pIC50-agg.min_assay_pIC50
agg['IC50_median_nM']=10**(9-agg.pIC50)
metadata=good.groupby('standardized_smiles',as_index=False).agg(
    molecule_chembl_ids=('Molecule ChEMBL ID',join_unique),
    assay_chembl_ids=('Assay ChEMBL ID',join_unique),
    document_chembl_ids=('Document ChEMBL ID',join_unique),
    n_raw_records=('pIC50','size'))
agg=agg.merge(metadata,on='standardized_smiles',how='left')
print('Eligible measurements:',len(good),
      '| molecule-assay groups:',len(per_assay),
      '| unique structures after aggregation:',len(agg))
display(agg[['standardized_smiles','pIC50','IC50_median_nM','n_assays','n_raw_records']].head(5))

Eligible measurements: 2932 | molecule-assay groups: 2922 | unique structures after aggregation: 2827


,standardized_smiles,pIC50,IC50_median_nM,n_assays,n_raw_records
0,C#CCNC(=O)CC1N=C(c2ccc(Cl)cc2)c2cc(OC)ccc2-c2c...,6.677781,210.0,1,1
1,C#CCSc1nc2nc(C)cc(Nc3cc(-c4ccccc4)[nH]n3)n2n1,5.667562,2150.0,1,1
2,C#Cc1cc(-c2cc(S(C)(=O)=O)ccc2OCC2CC2)cn(C)c1=O,5.560667,2750.0,1,1
3,C#Cc1cccc2c1O[C@H](C)CC(=O)N2,5.051391,8884.0,1,1
4,C#Cc1sc2c(c1Cc1ccccc1)COCc1nnc(C)n1-2,6.677781,210.0,1,1


## 7. Flag cross-assay disagreement; finalize model-ready compounds

A compound whose assay-median pIC₅₀ values differ by **more than 1.0 log unit** is held out for scientific review (not automatically averaged into training). Export **both** the all-compounds aggregation audit and the filtered model-ready table. These exclusions are based on assay inconsistency, not the predictive model.

In [7]:
agg['decision']=np.where(agg.interassay_pIC50_range>1,
    'review_assay_disagreement_gt_1log','model_ready_automated')
agg.to_csv(AUDIT_DIR/'BD1_compound_aggregation_audit.csv',index=False)
final=agg.loc[agg.decision=='model_ready_automated'].copy()
final.to_csv(DATA_DIR/'BRD4_BD1_QSAR_model_ready_2816.csv',index=False)
print('Aggregated compounds:',len(agg),
      '| flagged for review:',int(agg.decision.ne('model_ready_automated').sum()),
      '| model-ready:',len(final))
display(agg.loc[agg.decision!='model_ready_automated',
     ['molecule_chembl_ids','n_assays','interassay_pIC50_range','decision']].head(15))

Aggregated compounds: 2827 | flagged for review: 11 | model-ready: 2816


,molecule_chembl_ids,n_assays,interassay_pIC50_range,decision
627,CHEMBL1232461,4,1.168792,review_assay_disagreement_gt_1log
744,CHEMBL4524758,2,1.357935,review_assay_disagreement_gt_1log
746,CHEMBL3987016,6,2.024134,review_assay_disagreement_gt_1log
1271,CHEMBL4785363,2,1.199365,review_assay_disagreement_gt_1log
1530,CHEMBL2179387,3,1.378442,review_assay_disagreement_gt_1log
1966,CHEMBL4159382,2,1.550353,review_assay_disagreement_gt_1log
2392,CHEMBL4241205,4,1.261659,review_assay_disagreement_gt_1log
2487,CHEMBL1828979,3,2.022222,review_assay_disagreement_gt_1log
2560,CHEMBL3581647,5,1.293891,review_assay_disagreement_gt_1log
2564,CHEMBL1957266,12,1.959571,review_assay_disagreement_gt_1log


## 8. Reproducibility and final integrity checks

Check each transition and verify the final CSV contains exactly one row per molecule, with no missing pIC₅₀ or invalid potency values. Print a concise audit summary. **The exact 2,816 count is an integrity check for this source snapshot, not a universal criterion for future ChEMBL releases.**

In [8]:
assert len(raw)==10919, 'Input extract changed: inspect new raw source before accepting counts.'
assert len(assays)==685
assert int(assay_audit.decision.eq('accept_BD1_biochemical').sum())==228
assert len(eligible)==2932 and len(good)==2932
assert len(agg)==2827 and int(agg.decision.ne('model_ready_automated').sum())==11
assert len(final)==2816
assert final.standardized_smiles.is_unique
assert final.pIC50.notna().all() and np.isfinite(final.pIC50).all()
assert final.IC50_median_nM.gt(0).all()
reloaded=pd.read_csv(DATA_DIR/'BRD4_BD1_QSAR_model_ready_2816.csv')
assert len(reloaded)==len(final) and reloaded.standardized_smiles.is_unique
summary={
 'source_file':INPUT_CSV.name,
 'source_note':'ChEMBL-derived exact-IC50 extract (not original unfiltered raw ZIP)',
 'input_exact_IC50_records':len(raw),
 'input_assays':len(assays),
 'assay_decisions':assay_audit.decision.value_counts().to_dict(),
 'record_decisions':raw.eligibility.value_counts().to_dict(),
 'eligible_records':len(eligible),
 'eligible_valid_structures':len(good),
 'unique_standardized_compounds':len(agg),
 'compounds_disagreement_review':int(agg.decision.ne('model_ready_automated').sum()),
 'model_ready_compounds':len(final),
 'domain_review':'Automated conservative domain filtering; confirm construct labels against source publications.',
 'aggregation':'Median pIC50 within assay, then median across assay medians.'}
(AUDIT_DIR/'summary.json').write_text(json.dumps(summary,indent=2))
print('FINAL CHECK PASSED —',len(reloaded),'unique structures; no missing potency or malformed rows.')
display(pd.DataFrame([
 ('Input exact-IC50 records',len(raw)),
 ('Assays reviewed',len(assays)),
 ('Accepted BD1 assays',int(assay_audit.decision.eq('accept_BD1_biochemical').sum())),
 ('Eligible BD1 activity records',len(eligible)),
 ('Aggregated compounds',len(agg)),
 ('Compounds held for inter-assay review',int(agg.decision.ne('model_ready_automated').sum())),
 ('Final model-ready compounds',len(reloaded))],columns=['Stage','Count']))
print('Exports:',(DATA_DIR/'BRD4_BD1_QSAR_model_ready_2816.csv').as_posix(),
      'and audit CSVs in',AUDIT_DIR.as_posix())

FINAL CHECK PASSED — 2816 unique structures; no missing potency or malformed rows.


,Stage,Count
0,Input exact-IC50 records,10919
1,Assays reviewed,685
2,Accepted BD1 assays,228
3,Eligible BD1 activity records,2932
4,Aggregated compounds,2827
5,Compounds held for inter-assay review,11
6,Final model-ready compounds,2816


Exports: data/BRD4_BD1_QSAR_model_ready_2816.csv and audit CSVs in audit


## Interpretation and limitations

The resulting dataset contains **2,816 unique standardized structures** with positive, traceable pIC₅₀ targets. **2,932** activity records entered aggregation, and **11** compounds were flagged because their assay medians differed by over one pIC₅₀ unit. The **228 accepted assays** were assigned by conservative text/format rules; these assignments remain subject to independent manual verification of protein domains and assay methods. This preprocessing does **not** establish predictive accuracy, drug inhibition, or biological confirmation.

Keep the model-ready data, complete assay decisions, record-level decisions, and compound-level audit together when sharing this notebook.